The "**Synthetic Healthcare Dataset**: Demographics, Conditions, Treatments, and Outcomes for Research and Analysis" is a complete synthesis of realistic but fictitious data that represents various aspects of healthcare. The database contains data about the patient demographics: age, gender, and region, as well as the medical conditions diagnosed, the treatments administered, and the outcomes observed.

The dataset has been created to resemble actual healthcare situations and can be used for research and analysis in the healthcare field. Researchers, data scientists, and healthcare professionals can use this dataset to discover the patterns, trends, and correlations related to disease prevalence, treatment effectiveness, patient outcomes, and other aspects. Besides, it is a good source for creating and testing models designed to enhance healthcare decision-making and patient care.

Through the collection of a wide variety of data, including patient characteristics, medical conditions, treatments, and outcomes, this synthetic dataset provides a multifaceted base for conducting numerous analyses and experiments in the area of healthcare analytics.

In [23]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

df = pd.read_csv("/kaggle/input/datasets/divyabhavana/synthetic-healthcare-dataset/medical_data.csv")
df.head()

,Patient_ID,Age,Gender,Medical_Condition,Treatment,Outcome,Insurance_Type,Income,Region,Smoking_Status,Admission_Type,Hospital_ID,Length_of_Stay
0,1,77,Female,Chronic Obstructive,Dialysis,Stable,Public,77444,North,Former smoker,Urgent,3173,20
1,2,62,Female,Obesity,Physical therapy,Improved,Public,19367,West,Non-smoker,Urgent,65671,4
2,3,77,Male,Hypertension,Inhaler therapy,Improved,Medicare,16054,North,Non-smoker,Urgent,96914,3
3,4,41,Female,Alzheimer's Disease,Medication C,Worsened,Medicare,54371,West,Non-smoker,Emergency,15732,11
4,5,82,Male,Alzheimer's Disease,Chemotherapy,Stable,Private,55489,West,Non-smoker,Emergency,98232,2


**To make this code highly reusable across Classification, Regression, Clustering, and Anomaly Detection, the best approach is to build an Object-Oriented ML Pipeline.**

**By structuring the code into a Python class**, we can create a central "**engine**." Depending on the parameters you pass into it (like task_type and target_col), the pipeline will automatically adjust its preprocessing, model selection, and evaluation metrics.

Here is the modular code designed specifically to run in a Kaggle Notebook environment.

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, mean_squared_error, r2_score

class HealthcareAIPipeline:
    def __init__(self, filepath, task_type='classification', target_col=None):
        """
        task_type options: 'classification', 'regression', 'clustering', 'anomaly'
        """
        self.filepath = filepath
        self.task_type = task_type
        self.target_col = target_col
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        
    def load_and_clean_data(self):
        # 1. Load data (Kaggle standard path format)
        self.df = pd.read_csv(self.filepath)
        
        # 2. Drop identifiers that don't hold predictive value
        cols_to_drop = [col for col in ['Name', 'Patient ID', 'Hospital'] if col in self.df.columns]
        self.df = self.df.drop(columns=cols_to_drop, errors='ignore')
        
        # 3. Handle missing values (simple forward fill for synthetic data)
        self.df = self.df.ffill().bfill()
        
    def preprocess_data(self):
        self.load_and_clean_data()
        
        # Separate features (X) and target (y) if applicable
        if self.task_type in ['classification', 'regression']:
            if self.target_col not in self.df.columns:
                raise ValueError(f"Target column '{self.target_col}' not found in dataset.")
            
            X = self.df.drop(columns=[self.target_col])
            y = self.df[self.target_col]
            
            # Encode Target if Classification
            if self.task_type == 'classification':
                le = LabelEncoder()
                y = le.fit_transform(y)
                self.target_encoder = le
        else:
            # Unsupervised learning (Clustering / Anomaly) uses all features
            X = self.df.copy()
            y = None
            
        # Encode categorical features
        cat_cols = X.select_dtypes(include=['object']).columns
        X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
        
        # Scale numerical features
        self.feature_names = X_encoded.columns
        X_scaled = self.scaler.fit_transform(X_encoded)
        
        return X_scaled, y
        
    def train_and_evaluate(self):
        X, y = self.preprocess_data()
        
        # Supervised Learning
        if self.task_type in ['classification', 'regression']:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            if self.task_type == 'classification':
                self.model = RandomForestClassifier(n_estimators=100, random_state=42)
                self.model.fit(X_train, y_train)
                preds = self.model.predict(X_test)
                
                print("--- Classification Report ---")
                # Decode labels back to original text for readability
                target_names = self.target_encoder.classes_.astype(str)
                print(classification_report(y_test, preds, target_names=target_names))
                
            elif self.task_type == 'regression':
                self.model = RandomForestRegressor(n_estimators=100, random_state=42)
                self.model.fit(X_train, y_train)
                preds = self.model.predict(X_test)
                
                mse = mean_squared_error(y_test, preds)
                r2 = r2_score(y_test, preds)
                print("--- Regression Metrics ---")
                print(f"Mean Squared Error: {mse:.2f}")
                print(f"R2 Score: {r2:.2f}")
                
        # Unsupervised Learning
        elif self.task_type == 'clustering':
            print("--- K-Means Clustering ---")
            self.model = KMeans(n_clusters=4, random_state=42, n_init=10)
            clusters = self.model.fit_predict(X)
            self.df['Cluster'] = clusters
            print(f"Assigned {len(self.df)} patients to 4 distinct clusters.")
            print(self.df['Cluster'].value_counts())
            
        elif self.task_type == 'anomaly':
            print("--- Anomaly Detection (Fraud/Outliers) ---")
            self.model = IsolationForest(contamination=0.05, random_state=42)
            anomalies = self.model.fit_predict(X)
            # Isolation forest outputs -1 for anomalies, 1 for normal
            self.df['Is_Anomaly'] = np.where(anomalies == -1, True, False)
            anomaly_count = self.df['Is_Anomaly'].sum()
            print(f"Detected {anomaly_count} anomalous patient records out of {len(self.df)}.")

**How to use this code?**  

Because of the object-oriented design, you don't need to rewrite your data cleaning or encoding logic. You just instantiate the class with different parameters.

Here is how you would trigger the different use cases in your subsequent Kaggle cells:


In [25]:
# Change the filepath to match your exact Kaggle directory
FILE_PATH = '/kaggle/input/datasets/divyabhavana/synthetic-healthcare-dataset/medical_data.csv'

**1. Predicting Outcomes (Classification)**  
Assuming the dataset has a column named Test Results/Outcome (often "Normal", "Abnormal", "Inconclusive") or a generic Outcome column.  

You can train a model to predict the final health outcome for a patient (e.g., recovered, readmitted, or deceased) based on their initial conditions and demographic data.

**How it works:** The model analyzes patterns between the patient's profile (age, gender, pre-existing conditions), the treatment administered, and the historical outcome.

**Algorithms:** **Random Forest**, XGBoost, Logistic Regression, or Support Vector Machines (SVM).

**Use Case:** Helping doctors flag high-risk patients who might need more aggressive observation.

In [26]:

classifier = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='classification', 
    target_col='Outcome' # Replace with actual outcome column name
)
classifier.train_and_evaluate()

--- Classification Report ---
              precision    recall  f1-score   support

    Improved       0.34      0.50      0.40        64
      Stable       0.44      0.27      0.33        75
    Worsened       0.25      0.25      0.25        61

    accuracy                           0.34       200
   macro avg       0.34      0.34      0.33       200
weighted avg       0.35      0.34      0.33       200



----------------------------------------------------------------------

**2. Length of Stay (LOS) Prediction (Regression)**  
Hospital logistics rely heavily on predicting how long a patient will occupy a bed. You can train an AI model to estimate the exact number of days a patient will stay in the facility.

**How it works:** You use the admission date, diagnosis, and demographic details as input features to predict a continuous number (days).

**Algorithms:** **Linear Regression**, Gradient Boosting Regressors, or Deep Neural Networks.

**Use Case:** Optimizing hospital resource allocation, bed management, and staff scheduling.

**Length of Stay / Cost Estimation** 
Assuming there is a numeric column like Billing Amount or Days Hospitalized (**Length_of_Stay**).

In [27]:
regressor = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='regression', 
    target_col='Length_of_Stay' # Replace with actual numeric column
)
regressor.train_and_evaluate()

--- Regression Metrics ---
Mean Squared Error: 41.46
R2 Score: -0.11


--------------------------------------------

**3. Treatment Recommendation Systems (Classification / Recommendation)**  
By leveraging the "Treatments" and "Outcomes" features, you can build an AI assistant that suggests the most historically effective treatment plan for a specific condition.

**How it works:** The model learns which treatments yielded the highest success rates for specific conditions across different demographic groups.

**Algorithms:** K-Nearest Neighbors (KNN), Decision Trees, or Collaborative Filtering techniques.

**Use Case:** Providing clinical decision support to physicians by recommending evidence-based treatments.  


In [28]:
classifier = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='classification', 
    target_col='Treatment' # Replace with actual outcome column name
)
classifier.train_and_evaluate()

--- Classification Report ---
                    precision    recall  f1-score   support

Bone density tests       0.07      0.08      0.07        13
      Chemotherapy       0.29      0.24      0.26        17
          Dialysis       0.08      0.07      0.07        15
   Dietary changes       0.08      0.08      0.08        12
Dietary counseling       0.10      0.19      0.13        16
Immunosuppressants       0.15      0.10      0.12        20
   Inhaler therapy       0.08      0.07      0.08        14
      Medication A       0.08      0.11      0.09         9
      Medication B       0.13      0.25      0.17        16
      Medication C       0.18      0.11      0.14        18
  Memory exercises       0.05      0.06      0.05        18
  Physical therapy       0.00      0.00      0.00        18
           Therapy       0.00      0.00      0.00        14

          accuracy                           0.10       200
         macro avg       0.10      0.10      0.10       200
      we

----------------------

**4. Healthcare Cost and Billing Estimation (Regression)**  
If the dataset contains financial or billing data (common in synthetic healthcare data), you can build a model to estimate the cost of care at the time of admission.

**How it works:** The AI correlates the medical condition, expected treatment, and demographics with final billing amounts.

**Algorithms:** Ridge/Lasso Regression, Random Forest Regressor.

**Use Case:** Providing patients with transparent cost estimates and helping insurance providers predict claim amounts.  


In [29]:
regressor = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='regression', 
    target_col='Income' # Replace with actual numeric column such as Billing Amount
)
regressor.train_and_evaluate()

--- Regression Metrics ---
Mean Squared Error: 987739808.82
R2 Score: -0.13


-----------------------

**5. Patient Segmentation and Risk Stratification (Clustering)**  
Instead of predicting a specific target, you can use unsupervised learning to discover hidden patterns and group similar patients together.

**How it works:** The model clusters patients based on similarities in their demographics and medical conditions without needing predefined labels.

**Algorithms:** K-Means Clustering, DBSCAN, or Hierarchical Clustering.

**Use Case:** Identifying distinct patient personas (e.g., "high-risk chronic patients") to design targeted, preventative public health programs.

**Example: Patient Risk Stratification (Clustering)**
Because clustering is unsupervised, it groups patients based on all available features without needing a target column.

In [30]:
clusterer = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='clustering'
)
clusterer.train_and_evaluate()

--- K-Means Clustering ---
Assigned 1000 patients to 4 distinct clusters.
Cluster
1    460
3    408
2    114
0     18
Name: count, dtype: int64


---------------------------

**6. Medical Fraud or Anomaly Detection (Anomaly Detection)**  
Synthetic datasets are excellent for training models to spot outliers, which is a key task in auditing and compliance.

**How it works:** The AI learns what a "normal" patient record looks like. If a new record features a rare treatment for a common condition or an exorbitant billing amount, the model flags it.

**Algorithms:** Isolation Forest, One-Class SVM, or Autoencoders.

**Use Case:** Detecting data entry errors, irregular medical practices, or potential insurance billing fraud.  

**Example:** Fraud / Anomaly Detection (Anomaly)  
Finds the 5% (contamination rate set in the class) most unusual patient records.

In [31]:
anomaly_detector = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='anomaly'
)
anomaly_detector.train_and_evaluate()

--- Anomaly Detection (Fraud/Outliers) ---
Detected 50 anomalous patient records out of 1000.
